# DOVER (Disentangled Objective Video quality EvaluatoR) - 完整實作指南

這個 notebook 提供了 DOVER 模型的完整實作範例，包括：
- 環境設置
- 模型載入
- 視頻質量評估
- 結果視覺化
- 進階應用

**論文**: [Exploring Video Quality Assessment on User Generated Contents from Aesthetic and Technical Perspectives](https://arxiv.org/pdf/2211.04894v3)

**官方代碼**: [GitHub - DOVER](https://github.com/VQAssessment/DOVER)

## 1. 環境設置與依賴安裝

首先，我們需要安裝所有必要的依賴套件。

In [ ]:
# 檢查 GPU 是否可用
!nvidia-smi

In [ ]:
# 安裝必要的套件
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install decord  # 視頻解碼庫
!pip install timm    # PyTorch Image Models
!pip install opencv-python  # 視頻處理
!pip install matplotlib seaborn  # 視覺化
!pip install scipy scikit-learn  # 數據分析
!pip install tqdm  # 進度條

In [ ]:
# 匯入必要的庫
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 設置隨機種子以確保可重現性
torch.manual_seed(42)
np.random.seed(42)

# 檢查 CUDA 是否可用
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用設備: {device}")
if torch.cuda.is_available():
    print(f"GPU 名稱: {torch.cuda.get_device_name(0)}")
    print(f"GPU 記憶體: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

## 2. 下載 DOVER 代碼與預訓練模型

我們將從官方 GitHub 倉庫克隆代碼並下載預訓練模型。

In [ ]:
# 克隆 DOVER 官方代碼庫
import os
if not os.path.exists('DOVER'):
    !git clone https://github.com/VQAssessment/DOVER.git
    print("✓ DOVER 代碼庫克隆完成")
else:
    print("✓ DOVER 代碼庫已存在")

# 切換到 DOVER 目錄
%cd DOVER

In [ ]:
# 下載預訓練模型（如果官方提供的話）
# 這裡需要根據官方文檔更新下載鏈接
import requests
from pathlib import Path

# 創建模型目錄
Path("pretrained_models").mkdir(exist_ok=True)

# 下載 ConvNeXt backbone 預訓練權重
convnext_url = "https://dl.fbaipublicfiles.com/convnext/convnext_tiny_1k_224_ema.pth"
convnext_path = "pretrained_models/convnext_tiny_1k_224_ema.pth"

if not os.path.exists(convnext_path):
    print("下載 ConvNeXt 預訓練權重...")
    !wget -O {convnext_path} {convnext_url}
    print("✓ ConvNeXt 權重下載完成")
else:
    print("✓ ConvNeXt 權重已存在")

## 3. DOVER 模型架構實現

這裡我們實現 DOVER 的核心架構，包括美學分支和技術分支。

In [ ]:
# DOVER 模型架構概述
print("""
DOVER 架構概述：
================

┌─────────────────────────────────────────────┐
│              輸入視頻                        │
└─────────────────────────────────────────────┘
                    │
        ┌───────────┴───────────┐
        │                       │
        ▼                       ▼
┌──────────────┐        ┌──────────────┐
│  美學分支     │        │  技術分支     │
│              │        │              │
│ - 下採樣      │        │ - 保持分辨率  │
│ - 稀疏幀採樣  │        │ - 連續幀採樣  │
│ - ConvNeXt   │        │ - ConvNeXt   │
│ - 語義特徵    │        │ - 失真特徵    │
└──────────────┘        └──────────────┘
        │                       │
        ▼                       ▼
   美學評分 (SA)           技術評分 (ST)
        │                       │
        └───────────┬───────────┘
                    ▼
           加權融合 (0.428*SA + 0.572*ST)
                    │
                    ▼
              整體質量評分 (MOS)
""")

In [ ]:
# 簡化的 DOVER 模型實現（用於示範）
import timm

class SimplifiedDOVER(nn.Module):
    """
    簡化版的 DOVER 模型，用於教學和演示
    
    架構：
    - 兩個獨立的 ConvNeXt backbone（美學分支和技術分支）
    - 各自的質量預測頭
    - 加權融合機制
    """
    
    def __init__(self, aesthetic_weight=0.428, technical_weight=0.572):
        super().__init__()
        
        # 美學分支 - 使用較小的輸入尺寸
        self.aesthetic_backbone = timm.create_model(
            'convnext_tiny',
            pretrained=True,
            num_classes=0  # 移除分類頭
        )
        
        # 技術分支 - 使用完整的輸入尺寸
        self.technical_backbone = timm.create_model(
            'convnext_tiny',
            pretrained=True,
            num_classes=0
        )
        
        # 獲取特徵維度
        feature_dim = self.aesthetic_backbone.num_features
        
        # 美學質量預測頭
        self.aesthetic_head = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1)
        )
        
        # 技術質量預測頭
        self.technical_head = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1)
        )
        
        # 融合權重
        self.aesthetic_weight = aesthetic_weight
        self.technical_weight = technical_weight
        
    def forward(self, aesthetic_input, technical_input):
        """
        前向傳播
        
        Args:
            aesthetic_input: 美學分支輸入 (B, T, C, H, W) - 較小分辨率
            technical_input: 技術分支輸入 (B, T, C, H, W) - 完整分辨率
        
        Returns:
            aesthetic_score: 美學評分
            technical_score: 技術評分
            overall_score: 整體評分
        """
        # 處理美學分支
        B, T, C, H, W = aesthetic_input.shape
        aesthetic_input = aesthetic_input.view(B * T, C, H, W)
        aesthetic_features = self.aesthetic_backbone(aesthetic_input)
        aesthetic_features = aesthetic_features.view(B, T, -1).mean(dim=1)  # 時序池化
        aesthetic_score = self.aesthetic_head(aesthetic_features)
        
        # 處理技術分支
        B, T, C, H, W = technical_input.shape
        technical_input = technical_input.view(B * T, C, H, W)
        technical_features = self.technical_backbone(technical_input)
        technical_features = technical_features.view(B, T, -1).mean(dim=1)  # 時序池化
        technical_score = self.technical_head(technical_features)
        
        # 融合得到整體評分
        overall_score = (self.aesthetic_weight * aesthetic_score + 
                        self.technical_weight * technical_score)
        
        return aesthetic_score, technical_score, overall_score

print("✓ DOVER 模型類別定義完成")

## 4. 視頻預處理工具

實現視頻讀取、幀採樣和預處理功能。

In [ ]:
import cv2
from torchvision import transforms

class VideoPreprocessor:
    """
    視頻預處理類
    
    功能：
    - 視頻讀取
    - 幀採樣（稀疏採樣用於美學，連續採樣用於技術）
    - 分辨率調整
    - 正規化
    """
    
    def __init__(self, 
                 aesthetic_size=128,
                 technical_size=224,
                 num_frames=8):
        
        self.aesthetic_size = aesthetic_size
        self.technical_size = technical_size
        self.num_frames = num_frames
        
        # ImageNet 標準化參數
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
        
    def read_video(self, video_path):
        """
        讀取視頻文件
        
        Args:
            video_path: 視頻文件路徑
            
        Returns:
            frames: 幀列表
        """
        cap = cv2.VideoCapture(video_path)
        frames = []
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            # BGR to RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        
        cap.release()
        return frames
    
    def sample_frames(self, frames, mode='sparse'):
        """
        幀採樣
        
        Args:
            frames: 完整幀列表
            mode: 'sparse' 用於美學分支, 'dense' 用於技術分支
            
        Returns:
            sampled_frames: 採樣後的幀列表
        """
        total_frames = len(frames)
        
        if mode == 'sparse':
            # 稀疏採樣：均勻分佈地選取幀
            indices = np.linspace(0, total_frames - 1, self.num_frames, dtype=int)
        else:
            # 密集採樣：連續幀
            start_idx = max(0, (total_frames - self.num_frames) // 2)
            indices = range(start_idx, start_idx + self.num_frames)
            indices = [min(i, total_frames - 1) for i in indices]
        
        return [frames[i] for i in indices]
    
    def preprocess_frames(self, frames, target_size):
        """
        預處理幀
        
        Args:
            frames: 幀列表
            target_size: 目標尺寸
            
        Returns:
            tensor: 預處理後的張量 (T, C, H, W)
        """
        processed_frames = []
        
        for frame in frames:
            # 調整大小
            frame = cv2.resize(frame, (target_size, target_size))
            # 轉換為張量並正規化
            frame = torch.from_numpy(frame).float() / 255.0
            frame = frame.permute(2, 0, 1)  # HWC to CHW
            frame = self.normalize(frame)
            processed_frames.append(frame)
        
        return torch.stack(processed_frames)
    
    def process_video(self, video_path):
        """
        完整的視頻處理流程
        
        Args:
            video_path: 視頻文件路徑
            
        Returns:
            aesthetic_input: 美學分支輸入
            technical_input: 技術分支輸入
        """
        # 讀取視頻
        frames = self.read_video(video_path)
        print(f"讀取到 {len(frames)} 幀")
        
        # 美學分支：稀疏採樣 + 小尺寸
        aesthetic_frames = self.sample_frames(frames, mode='sparse')
        aesthetic_input = self.preprocess_frames(aesthetic_frames, self.aesthetic_size)
        
        # 技術分支：密集採樣 + 原始尺寸
        technical_frames = self.sample_frames(frames, mode='dense')
        technical_input = self.preprocess_frames(technical_frames, self.technical_size)
        
        return aesthetic_input.unsqueeze(0), technical_input.unsqueeze(0)

print("✓ 視頻預處理器定義完成")

## 5. 測試視頻評估

下載測試視頻並進行質量評估。

In [ ]:
# 創建測試視頻目錄
test_video_dir = Path("test_videos")
test_video_dir.mkdir(exist_ok=True)

# 下載測試視頻（使用公開的示例視頻）
print("準備測試視頻...")
print("請上傳您的測試視頻，或使用示例代碼下載公開視頻")

# 這裡可以添加視頻下載邏輯
# 或者使用 Google Colab 的文件上傳功能

# 示例：列出目錄中的視頻文件
video_files = list(test_video_dir.glob("*.mp4"))
print(f"找到 {len(video_files)} 個視頻文件")

In [ ]:
# 初始化模型和預處理器
print("初始化 DOVER 模型...")
model = SimplifiedDOVER().to(device)
model.eval()
print(f"✓ 模型已載入到 {device}")

preprocessor = VideoPreprocessor(
    aesthetic_size=128,
    technical_size=224,
    num_frames=8
)
print("✓ 預處理器已初始化")

# 模型參數統計
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n模型參數統計:")
print(f"總參數: {total_params:,}")
print(f"可訓練參數: {trainable_params:,}")

In [ ]:
def evaluate_video(video_path, model, preprocessor, device):
    """
    評估單個視頻的質量
    
    Args:
        video_path: 視頻文件路徑
        model: DOVER 模型
        preprocessor: 視頻預處理器
        device: 計算設備
        
    Returns:
        結果字典包含美學、技術和整體評分
    """
    print(f"\n正在評估視頻: {video_path}")
    print("-" * 60)
    
    # 預處理視頻
    aesthetic_input, technical_input = preprocessor.process_video(str(video_path))
    aesthetic_input = aesthetic_input.to(device)
    technical_input = technical_input.to(device)
    
    # 模型推理
    with torch.no_grad():
        aesthetic_score, technical_score, overall_score = model(
            aesthetic_input, 
            technical_input
        )
    
    # 轉換為標準分數範圍 (0-100)
    results = {
        'aesthetic': float(aesthetic_score.cpu().item()),
        'technical': float(technical_score.cpu().item()),
        'overall': float(overall_score.cpu().item())
    }
    
    # 顯示結果
    print(f"\n評估結果:")
    print(f"  美學評分: {results['aesthetic']:.3f}")
    print(f"  技術評分: {results['technical']:.3f}")
    print(f"  整體評分: {results['overall']:.3f}")
    print("-" * 60)
    
    return results

# 如果有測試視頻，進行評估
if len(video_files) > 0:
    test_video = video_files[0]
    results = evaluate_video(test_video, model, preprocessor, device)
else:
    print("請先上傳或下載測試視頻")

## 6. 結果視覺化

視覺化評估結果，幫助理解不同維度的質量評分。

In [ ]:
def visualize_results(results, video_name="測試視頻"):
    """
    視覺化視頻質量評估結果
    
    Args:
        results: 評估結果字典
        video_name: 視頻名稱
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # 子圖 1: 柱狀圖
    categories = ['美學評分', '技術評分', '整體評分']
    scores = [results['aesthetic'], results['technical'], results['overall']]
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    
    bars = axes[0].bar(categories, scores, color=colors, alpha=0.7, edgecolor='black')
    axes[0].set_ylabel('分數', fontsize=12)
    axes[0].set_title(f'{video_name} - 質量評分', fontsize=14, fontweight='bold')
    axes[0].set_ylim([min(scores) * 0.9, max(scores) * 1.1])
    axes[0].grid(axis='y', alpha=0.3, linestyle='--')
    
    # 在柱狀圖上添加數值標籤
    for bar, score in zip(bars, scores):
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{score:.3f}',
                    ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    # 子圖 2: 雷達圖
    # 將分數正規化到 0-1 範圍以便比較
    normalized_scores = [
        (results['aesthetic'] - min(scores)) / (max(scores) - min(scores) + 1e-6),
        (results['technical'] - min(scores)) / (max(scores) - min(scores) + 1e-6),
        (results['overall'] - min(scores)) / (max(scores) - min(scores) + 1e-6)
    ]
    
    # 計算角度
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    normalized_scores += normalized_scores[:1]  # 閉合圖形
    angles += angles[:1]
    
    ax = plt.subplot(122, projection='polar')
    ax.plot(angles, normalized_scores, 'o-', linewidth=2, color='#45B7D1')
    ax.fill(angles, normalized_scores, alpha=0.25, color='#45B7D1')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    ax.set_ylim(0, 1)
    ax.set_title('質量維度雷達圖', fontsize=14, fontweight='bold', pad=20)
    ax.grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # 打印詳細分析
    print("\n📊 詳細分析:")
    print("=" * 60)
    
    if results['aesthetic'] > results['technical']:
        print("✓ 美學評分高於技術評分")
        print("  → 該視頻在內容、構圖等方面表現較好")
        print("  → 但可能存在一些技術性失真（模糊、噪點等）")
    elif results['technical'] > results['aesthetic']:
        print("✓ 技術評分高於美學評分")
        print("  → 該視頻在清晰度、流暢度等技術方面表現較好")
        print("  → 但在內容吸引力、構圖等方面有提升空間")
    else:
        print("✓ 美學評分與技術評分相當")
        print("  → 該視頻在美學和技術兩方面保持平衡")
    
    print("\n建議:")
    if results['aesthetic'] < results['technical']:
        print("  - 考慮改善視頻構圖和內容呈現")
        print("  - 增強視覺吸引力和故事性")
    if results['technical'] < results['aesthetic']:
        print("  - 提高視頻拍攝或編碼質量")
        print("  - 減少壓縮失真和畫質損失")
    
    print("=" * 60)

# 如果有評估結果，進行視覺化
if 'results' in locals():
    visualize_results(results, video_name=test_video.name)
else:
    print("請先運行視頻評估")

## 7. 批量評估與比較

對多個視頻進行批量評估並比較結果。

In [ ]:
def batch_evaluate(video_dir, model, preprocessor, device):
    """
    批量評估目錄中的所有視頻
    
    Args:
        video_dir: 視頻目錄路徑
        model: DOVER 模型
        preprocessor: 視頻預處理器
        device: 計算設備
        
    Returns:
        results_dict: 所有視頻的評估結果
    """
    video_files = list(Path(video_dir).glob("*.mp4"))
    print(f"找到 {len(video_files)} 個視頻文件")
    
    results_dict = {}
    
    for video_path in tqdm(video_files, desc="批量評估"):
        try:
            results = evaluate_video(video_path, model, preprocessor, device)
            results_dict[video_path.name] = results
        except Exception as e:
            print(f"評估 {video_path.name} 時出錯: {e}")
            continue
    
    return results_dict

def compare_videos(results_dict):
    """
    比較多個視頻的評估結果
    
    Args:
        results_dict: 視頻評估結果字典
    """
    if len(results_dict) == 0:
        print("沒有可比較的結果")
        return
    
    # 準備數據
    video_names = list(results_dict.keys())
    aesthetic_scores = [results_dict[v]['aesthetic'] for v in video_names]
    technical_scores = [results_dict[v]['technical'] for v in video_names]
    overall_scores = [results_dict[v]['overall'] for v in video_names]
    
    # 創建比較圖表
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 子圖 1: 分組柱狀圖
    x = np.arange(len(video_names))
    width = 0.25
    
    axes[0, 0].bar(x - width, aesthetic_scores, width, label='美學', color='#FF6B6B', alpha=0.7)
    axes[0, 0].bar(x, technical_scores, width, label='技術', color='#4ECDC4', alpha=0.7)
    axes[0, 0].bar(x + width, overall_scores, width, label='整體', color='#45B7D1', alpha=0.7)
    
    axes[0, 0].set_xlabel('視頻', fontsize=11)
    axes[0, 0].set_ylabel('分數', fontsize=11)
    axes[0, 0].set_title('視頻質量比較', fontsize=13, fontweight='bold')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels([name[:15] + '...' if len(name) > 15 else name 
                                 for name in video_names], rotation=45, ha='right')
    axes[0, 0].legend()
    axes[0, 0].grid(axis='y', alpha=0.3, linestyle='--')
    
    # 子圖 2: 美學 vs 技術散點圖
    axes[0, 1].scatter(aesthetic_scores, technical_scores, s=200, alpha=0.6, 
                      c=overall_scores, cmap='viridis', edgecolors='black', linewidths=2)
    axes[0, 1].set_xlabel('美學評分', fontsize=11)
    axes[0, 1].set_ylabel('技術評分', fontsize=11)
    axes[0, 1].set_title('美學 vs 技術評分', fontsize=13, fontweight='bold')
    axes[0, 1].grid(alpha=0.3, linestyle='--')
    
    # 添加對角線
    min_score = min(min(aesthetic_scores), min(technical_scores))
    max_score = max(max(aesthetic_scores), max(technical_scores))
    axes[0, 1].plot([min_score, max_score], [min_score, max_score], 
                    'r--', alpha=0.5, label='平衡線')
    axes[0, 1].legend()
    
    # 為每個點添加標籤
    for i, name in enumerate(video_names):
        axes[0, 1].annotate(name[:10], 
                           (aesthetic_scores[i], technical_scores[i]),
                           fontsize=8, alpha=0.7)
    
    # 子圖 3: 分數分布直方圖
    all_scores = aesthetic_scores + technical_scores + overall_scores
    axes[1, 0].hist([aesthetic_scores, technical_scores, overall_scores],
                    bins=10, label=['美學', '技術', '整體'],
                    color=['#FF6B6B', '#4ECDC4', '#45B7D1'],
                    alpha=0.6, edgecolor='black')
    axes[1, 0].set_xlabel('分數', fontsize=11)
    axes[1, 0].set_ylabel('頻率', fontsize=11)
    axes[1, 0].set_title('分數分布', fontsize=13, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(axis='y', alpha=0.3, linestyle='--')
    
    # 子圖 4: 統計摘要
    axes[1, 1].axis('off')
    
    summary_text = f"""
    📊 統計摘要
    {'=' * 40}
    
    總視頻數: {len(video_names)}
    
    美學評分:
      平均: {np.mean(aesthetic_scores):.3f}
      標準差: {np.std(aesthetic_scores):.3f}
      範圍: [{np.min(aesthetic_scores):.3f}, {np.max(aesthetic_scores):.3f}]
    
    技術評分:
      平均: {np.mean(technical_scores):.3f}
      標準差: {np.std(technical_scores):.3f}
      範圍: [{np.min(technical_scores):.3f}, {np.max(technical_scores):.3f}]
    
    整體評分:
      平均: {np.mean(overall_scores):.3f}
      標準差: {np.std(overall_scores):.3f}
      範圍: [{np.min(overall_scores):.3f}, {np.max(overall_scores):.3f}]
    
    最佳視頻:
      美學: {video_names[np.argmax(aesthetic_scores)][:20]}
      技術: {video_names[np.argmax(technical_scores)][:20]}
      整體: {video_names[np.argmax(overall_scores)][:20]}
    """
    
    axes[1, 1].text(0.1, 0.5, summary_text, fontsize=10, 
                   family='monospace', verticalalignment='center')
    
    plt.tight_layout()
    plt.show()

print("✓ 批量評估函數已定義")
print("\n使用方式:")
print("  results_dict = batch_evaluate('test_videos', model, preprocessor, device)")
print("  compare_videos(results_dict)")

## 8. 個性化質量評估

根據不同的應用場景調整美學和技術權重。

In [ ]:
def personalized_evaluation(aesthetic_score, technical_score, 
                           scenario='balanced'):
    """
    個性化質量評估
    
    Args:
        aesthetic_score: 美學評分
        technical_score: 技術評分
        scenario: 應用場景
            - 'artistic': 強調美學（如短視頻、藝術內容）
            - 'professional': 強調技術（如專業視頻、教學內容）
            - 'balanced': 平衡（默認）
            
    Returns:
        personalized_score: 個性化評分
    """
    scenarios = {
        'artistic': (0.7, 0.3),      # 70% 美學, 30% 技術
        'professional': (0.3, 0.7),   # 30% 美學, 70% 技術
        'balanced': (0.5, 0.5),       # 50% 美學, 50% 技術
        'dover_default': (0.428, 0.572)  # DOVER 論文建議的權重
    }
    
    if scenario not in scenarios:
        print(f"未知場景 '{scenario}'，使用默認平衡模式")
        scenario = 'balanced'
    
    aesthetic_weight, technical_weight = scenarios[scenario]
    personalized_score = (aesthetic_weight * aesthetic_score + 
                         technical_weight * technical_score)
    
    return personalized_score, aesthetic_weight, technical_weight

def compare_scenarios(aesthetic_score, technical_score):
    """
    比較不同場景下的評分
    
    Args:
        aesthetic_score: 美學評分
        technical_score: 技術評分
    """
    scenarios = ['artistic', 'balanced', 'professional', 'dover_default']
    scenario_names = ['藝術導向', '平衡模式', '專業導向', 'DOVER 默認']
    
    results = []
    weights = []
    
    for scenario in scenarios:
        score, aes_w, tech_w = personalized_evaluation(
            aesthetic_score, technical_score, scenario
        )
        results.append(score)
        weights.append((aes_w, tech_w))
    
    # 視覺化比較
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # 子圖 1: 不同場景的評分
    colors = ['#FF6B6B', '#FFD93D', '#4ECDC4', '#45B7D1']
    bars = axes[0].barh(scenario_names, results, color=colors, alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('評分', fontsize=12)
    axes[0].set_title('不同場景下的質量評分', fontsize=14, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3, linestyle='--')
    
    # 添加數值標籤
    for bar, score in zip(bars, results):
        width = bar.get_width()
        axes[0].text(width, bar.get_y() + bar.get_height()/2.,
                    f'{score:.3f}',
                    ha='left', va='center', fontsize=10, fontweight='bold')
    
    # 子圖 2: 權重分配
    x = np.arange(len(scenario_names))
    aesthetic_weights = [w[0] for w in weights]
    technical_weights = [w[1] for w in weights]
    
    axes[1].bar(x, aesthetic_weights, 0.4, label='美學權重', 
               color='#FF6B6B', alpha=0.7, edgecolor='black')
    axes[1].bar(x, technical_weights, 0.4, bottom=aesthetic_weights,
               label='技術權重', color='#4ECDC4', alpha=0.7, edgecolor='black')
    
    axes[1].set_ylabel('權重', fontsize=12)
    axes[1].set_title('不同場景的權重分配', fontsize=14, fontweight='bold')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(scenario_names, rotation=45, ha='right')
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.show()
    
    # 打印詳細信息
    print("\n🎯 個性化評估結果:")
    print("=" * 70)
    print(f"原始評分: 美學 = {aesthetic_score:.3f}, 技術 = {technical_score:.3f}")
    print("\n不同場景下的評分:")
    for name, score, (aes_w, tech_w) in zip(scenario_names, results, weights):
        print(f"  {name:12s}: {score:.3f}  (美學權重: {aes_w:.1%}, 技術權重: {tech_w:.1%})")
    print("=" * 70)

# 示例使用
if 'results' in locals():
    print("\n比較不同應用場景:")
    compare_scenarios(results['aesthetic'], results['technical'])
else:
    print("請先運行視頻評估以獲取評分")

## 9. 結論與未來工作

本 notebook 展示了 DOVER 模型的核心概念和實作方法。主要內容包括：

### 完成的工作
1. ✅ 實現了 DOVER 的雙分支架構（美學 + 技術）
2. ✅ 開發了視頻預處理和幀採樣工具
3. ✅ 實現了單視頻和批量視頻評估功能
4. ✅ 提供了豐富的結果視覺化方法
5. ✅ 展示了個性化質量評估的應用

### 實際應用場景
- **視頻平台**: 自動評估用戶上傳內容的質量
- **內容推薦**: 根據質量評分優化推薦算法
- **視頻編輯**: 為創作者提供質量反饋
- **廣告投放**: 評估廣告素材質量
- **壓縮優化**: 評估不同壓縮參數的影響

### 局限性
- 本實現為簡化版本，實際 DOVER 模型更複雜
- 需要大量標註數據進行訓練
- 計算資源需求較高
- 對某些特定類型的視頻可能不夠準確

### 未來改進方向
1. 🚀 整合真實的預訓練權重
2. 🚀 添加更多的視頻增強功能
3. 🚀 支持實時視頻質量評估
4. 🚀 開發輕量化模型版本
5. 🚀 結合音頻質量進行多模態評估

### 相關資源
- 📄 論文: https://arxiv.org/pdf/2211.04894v3
- 💻 官方代碼: https://github.com/VQAssessment/DOVER
- 📊 數據集: DIVIDE-3k, LSVQ, KoNViD-1k

---

**感謝使用本教程！如有問題或建議，歡迎提出。**